# ⚙️ PySpark Transformations in ADF: Mapping Data Flows

## What are Mapping Data Flows?
In Azure Data Factory, **Data Flows** provide a low-code/no-code interface to perform complex data transformations. Behind the scenes, ADF translates your visual drag-and-drop logic into **PySpark code** and executes it on a managed Spark cluster. We will use this to clean our raw data (Bronze layer) and move it to our refined layer (Silver layer) using Delta Lake architecture.

*(Note: Before starting, ensure you go to your Azure Storage Account and create a new container or folder named `silver`).*

## ➕ Step 1: Create a Data Flow and Enable Debug
1. On the left menu under your Author tab, click the three dots next to **Data Flows** and select **New data flow**.
2. Rename it (e.g., `Silver_Transformations`).
3. **Turn on Data Flow Debug:** At the top of the canvas, toggle the **Data flow debug** button. 
   * *Explanation:* This spins up a temporary Spark cluster (taking ~5 minutes). It allows you to preview your data at every single transformation step, which is critical for debugging. Keep in mind that this compute is expensive, so only turn it on when actively developing.

## ➕ Step 2: Add Source Data (Bronze Layer)

Instead of creating just one source, we will add all 5 source streams (`dim_airline`, `dim_flight`, `dim_passenger`, `fact_bookings`, and `dim_airport`) into this single Data Flow to consolidate our transformations. 

1. Click **Add Source** on the canvas.
2. **Output stream name**: Rename the source stream to exactly match the table you are pulling (e.g., `dim_airline`). 
3. **Dataset Creation**: Click **+ New** to create a dedicated dataset on the fly.
   - Choose the format corresponding to your file (**CSV** for on-prem, **Parquet** for SQL, or **JSON** for GitHub).
   - **Name**: Use the exact naming convention from the video (e.g., `DS_dim_flight_source`, `DS_dim_passenger_source`, `DS_fact_source`, or `DS_dim_airport`). 
   - **Linked service**: Select your ADLS Gen2 linked service.
   - **File path**: Browse to your `bronze` container, open the corresponding folder (`onprem`, `SQL`, or `GitHub`), and select the file. 
   - *Crucial for CSVs:* Ensure **First row as header** is checked. Click **OK**.
4. Go to the **Projection** tab and click **Import projection**.
   - *Crucial Step:* You must import the projection to pull in the schema (column names and data types). As the presenter notes later, without this, you won't be able to select specific columns in downstream transformations.
5. Go to the **Data preview** tab and click **Refresh** to visually verify your Spark cluster is reading the data correctly.
6. **Repeat** steps 1-5 for the remaining 4 sources so all 5 are loaded onto your canvas.

## ➕ Step 3: Apply Common Transformations
Click the `+` icon next to your source to add transformation activities. Here are the most common ones used in the project:

### 1. Derived Column (Create or Modify Columns)
Used to generate new columns or update existing ones using ADF's expression builder.
* **Uppercase conversion:** Modify the `country` column using the expression `upper(country)`.
* **String Replacement (Regex):** Standardize values. For example, convert 'M' to 'Male' using `regex_replace(gender_flag, 'M', 'Male')`.
* **Splitting Strings:** Extract the first name from a full name column using the expression `split(full_name, ' ')[1]`. *(Note: ADF arrays are 1-indexed, so `[1]` grabs the first element).*

### 2. Select (Rename or Drop Columns)
Acts like a SQL `SELECT` statement.
* Use this to rename columns (e.g., rename `gender` to `gender_flag`).
* Use the trash can icon to drop columns you no longer need in the Silver layer (e.g., dropping redundant IDs or raw timestamp columns).

### 3. Filter
Used to keep only the rows that meet a specific condition.
* **Expression example:** `age > 25` to filter out passengers 25 and under.
* You can chain conditions using logical operators, e.g., `age > 25 && country != 'Mexico'`.

### 4. Cast (Change Data Types)
Used to standardize data types (e.g., converting decimals to integers to avoid precision errors).
* Select the column (e.g., `ticket_cost`) and change its type from `decimal` to `integer`.

## ➕ Step 4: The 'Alter Row' Activity (Crucial for Upserts)
Before writing data to your destination, you must explicitly tell Spark *how* to write the data (Insert, Update, Delete, or Upsert). 
1. Click the `+` icon and add an **Alter Row** activity.
2. **Condition:** Add a condition for `Upsert if`. 
3. **Expression:** Since we want to evaluate all incoming rows, we provide a pseudo-condition that is always true, such as `1 == 1`. 
   * *Explanation:* This simply passes all records through to the Sink, where the primary key logic will actually determine if it's a new insert or an update.

## ➕ Step 5: Sink (Writing to Delta Lake)
Now we write the transformed data into the Silver layer as a **Delta Table**.
1. Click the `+` icon after Alter Row and add a **Sink**.
2. **Dataset:** Instead of a standard dataset, choose **Inline Dataset** and select **Delta**.
   * *Explanation:* Delta Lake format provides ACID transactions and allows for upserts (merging data). It is required for modern Lakehouse architectures.
3. **Settings:** Browse to your Silver container and define the folder path (e.g., `silver/dim_passenger`).
4. **Update Method:** Check **Allow Upsert**.
5. **Key Columns:** Select the primary key column (e.g., `passenger_id`). The Spark cluster uses this key to check if the row already exists in the Delta table (triggering an update) or if it's new (triggering an insert).

## ➕ Step 6: Execute the Data Flow in a Pipeline
Data flows cannot run on their own; they must be orchestrated by a pipeline.
1. Create a new pipeline (e.g., `Silver_Layer_Execution`).
2. Drag the **Data Flow** activity onto the canvas.
3. Select the data flow you just built in the settings.
4. Click **Debug** to run the pipeline using your active debug cluster, or click **Publish All** to save your transformations.